# 04 · Baseline Training

**Goals**
- Baseline segmentation model, scored end to end with local Panoptic Quality (`src/metrics.py`)
- Loss choice informed by class imbalance
- Settle the two questions notebook 02 left to training: **which labels** (each / union / vote / soft)
  and **which input** (`full` preprocessing vs `raw`)
- Cross-validated PQ (mean ± std over folds), error analysis, model size, training and inference time

**Formulation.** Binary semantic segmentation (filament vs background) at 1024 px; instances are the
connected components (8-connectivity) of the thresholded probability map, upsampled to 2048 px and
restricted to the disk. Touching filaments therefore merge and gaps split a filament: these are the
failure modes notebook 05 attacks, and this baseline measures how much they cost.

**Running.** Needs a GPU: run on Kaggle (GPU + Internet on). Every run is appended to
`artifacts/baseline_runs.csv` and skipped if already there, so the experiments can be spread over
several sessions (copy `artifacts/` and `checkpoints/` back from the Kaggle output).
`COS711_SMOKE=1` runs the whole notebook on a few images at 256 px on CPU, to check the pipeline.

In [ ]:
# --- Setup: works both locally and on Kaggle ---
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    # Pull the shared src/ code from GitHub (requires Internet ON in notebook settings)
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT))

from src.data import get_data_dir
DATA_DIR = get_data_dir()
print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)

try:
    import segmentation_models_pytorch  # noqa: F401
except ImportError:
    %pip install -q segmentation-models-pytorch

In [ ]:
import json, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from src.data import SITES, load_annotations, load_image
from src.preprocess import preprocess, find_disk, build_cache, load_preprocessed, disk_mask
from src.augment import AugConfig, NO_AUG
from src.dataset import FilamentDataset, model_input
from src.model import build_model, BCEDiceLoss, model_size
from src.train import make_loader, fit, predict, evaluate, summarise
from src.postprocess import to_instances
from src.metrics import match_instances, pq_from_counts, error_types

FIG_DIR = ROOT / "figures" / "04"
ART_DIR = ROOT / "artifacts"
RUN_DIR = ART_DIR / "runs"
CKPT_DIR = ROOT / "checkpoints"
for d in (FIG_DIR, RUN_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)

SMOKE = os.environ.get("COS711_SMOKE") == "1"   # tiny CPU run: checks the pipeline, numbers meaningless
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

S = 256 if SMOKE else 1024        # working resolution (notebook 01)
FULL = 2048                       # evaluation resolution = original images
EPOCHS = 1 if SMOKE else 30
EVAL_EVERY = 1 if SMOKE else 2    # validation PQ every n epochs; best epoch is kept
BATCH = 2 if SMOKE else 4
LR = 3e-4
WORKERS = 0 if SMOKE else min(4, os.cpu_count())
VAL_FOLD = 0
THRESHOLDS = (0.3, 0.4, 0.5, 0.6, 0.7)
MIN_AREAS = (0, 25, 50, 100, 200, 400)   # full-resolution px
CACHE_DIR = Path("/tmp/cos711_cache") if IN_KAGGLE else None   # None: ROOT/cache
RUNS_CSV = ART_DIR / ("baseline_runs_smoke.csv" if SMOKE else "baseline_runs.csv")
GEOMETRIC = AugConfig(contrast=(1.0, 1.0), haze_prob=0.0, blur_prob=0.0, sharpen_prob=0.0)

print("device:", DEVICE, torch.cuda.get_device_name() if DEVICE.type == "cuda" else "", "| smoke:", SMOKE)

## 1. Data

Cleaned annotations (notebook 02), the grouped, site-stratified folds (`artifacts/folds.csv`) and the
full-resolution disk geometry (`artifacts/image_stats.csv`). Fold 0 is the validation set for model
selection; folds 1–4 are only used for the final cross-validation (section 7).

In [ ]:
images, anns = load_annotations(DATA_DIR)
excluded = pd.read_csv(ART_DIR / "excluded_annotations.csv")
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)
folds = pd.read_csv(ART_DIR / "folds.csv")
img_stats = pd.read_csv(ART_DIR / "image_stats.csv").set_index("obs")
geo_full = img_stats[["cx", "cy", "radius"]]
degraded = set(folds.loc[folds.degraded, "obs"])

if SMOKE:   # a few observations per fold
    folds = folds.groupby("fold").head(6).reset_index(drop=True)

def split(fold):
    return folds.loc[folds.fold != fold, "obs"].tolist(), folds.loc[folds.fold == fold, "obs"].tolist()

train_obs, val_obs = split(VAL_FOLD)
print(f"train {len(train_obs)} obs / val {len(val_obs)} obs")
folds.assign(part=np.where(folds.fold == VAL_FOLD, "val", "train")).pivot_table(
    index="part", columns="site", values="obs", aggfunc="count")

In [ ]:
# Preprocessed inputs at working resolution, cached as float16 (both inputs compared below)
paths = images.drop_duplicates("obs").set_index("obs").path
for mode in ("full", "raw"):
    t0 = time.time()
    build_cache([paths[o] for o in folds.obs], [tuple(geo_full.loc[o]) for o in folds.obs],
                workers=os.cpu_count(), size=S, mode=mode, cache_dir=CACHE_DIR)
    print(f"{mode}: cache ready in {time.time() - t0:.0f} s")

### Input normalisation and class imbalance

The network input is `(x − centre) / scale` with a **fixed** centre and scale per input type (median and
robust std of on-disk pixels over the training images). Per-image standardisation would undo the
contrast augmentation of notebook 03. The filament fraction of disk pixels shows how imbalanced the problem is.

In [ ]:
probe = FilamentDataset(train_obs, images, anns, geo_full, label="union", size=S, cache_dir=CACHE_DIR)
sample = probe.obs[:: max(1, len(probe.obs) // 100)]

NORM = {}
for mode in ("full", "raw"):
    v = []
    for o in sample:
        g = tuple(c * S / FULL for c in geo_full.loc[o])
        x = load_preprocessed(paths[o], tuple(geo_full.loc[o]), S, mode, CACHE_DIR)
        v.append(x[disk_mask(x.shape, *g, 0.985)][::7])
    v = np.concatenate(v)
    med = float(np.median(v))
    NORM[mode] = (med, float(1.4826 * np.median(np.abs(v - med))))
print("normalisation (centre, scale):", {k: tuple(round(c, 4) for c in v) for k, v in NORM.items()})

fg = []
for o in sample:
    g = tuple(c * S / FULL for c in geo_full.loc[o])
    on_disk = disk_mask((S, S), *g)
    for m in probe.masks(o):
        fg.append((m[on_disk] > 0).mean())
fg = np.array(fg)
print(f"filament pixels per annotation set: {100 * fg.mean():.2f}% of the disk "
      f"(median {100 * np.median(fg):.2f}%, max {100 * fg.max():.2f}%) → about 1 : {1 / fg.mean():.0f}")

## 2. Model, loss and training

| Choice | Setting | Why |
|---|---|---|
| Architecture | U-Net, ResNet34 encoder (ImageNet weights, stem collapsed to 1 channel), `segmentation_models_pytorch` | Skip connections keep the 1–3 px detail filaments need; a pretrained encoder converges fast on ~560 training images |
| Loss | 0.5 · BCE + 0.5 · soft Dice (per image) | Filaments are ~0.1–0.5% of the disk (above): Dice does not reward predicting all background; BCE gives stable per-pixel gradients |
| Optimiser | AdamW, lr 3e-4, weight decay 1e-4, 1 epoch warmup + cosine decay, fp16 autocast | Standard fine-tuning recipe |
| Batch / epochs | 4 × 1024² / 30 epochs, validation PQ every 2 epochs, best epoch kept | Fits a 16 GB Kaggle GPU |
| Augmentation | `AugConfig()` from notebook 03 (geometric only for `raw`, whose intensities are not centred on 1) | Calibrated against real and test images |
| Post-processing | threshold, disk mask, connected components, drop components < `min_area` (full-res px) | Simplest way from pixels to instances; threshold and `min_area` are tuned on the validation fold |

PQ is computed at 2048 px against **every annotation set** of each validation observation (counts pooled),
the same protocol as the annotator comparisons in notebooks 01–02.

In [ ]:
def datasets(label, mode, fold, seed):
    tr_obs, va_obs = split(fold)
    aug = GEOMETRIC if mode == "raw" else AugConfig()
    kw = dict(images=images, anns=anns, geo_full=geo_full, label=label, mode=mode, size=S,
              norm=NORM[mode], cache_dir=CACHE_DIR)
    return FilamentDataset(tr_obs, aug=aug, seed=seed, **kw), FilamentDataset(va_obs, aug=NO_AUG, **kw)

def scorer(va):
    # validation helpers: GT at full resolution for every set, full-resolution disk per observation
    gt_fn = lambda o: va.masks(o, FULL)
    disks = {o: va.geo[o] for o in va.obs}
    return gt_fn, disks

def run(name, label="each", mode="full", fold=VAL_FOLD, seed=0, aug=None, epochs=EPOCHS):
    # Train one model, tune the post-processing on its validation fold, save everything, return a summary row
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va = datasets(label, mode, fold, seed)
    if aug is not None:
        tr.aug = aug; tr.reseed(seed)
    gt_fn, disks = scorer(va)
    va_dl = make_loader(va, 2 * BATCH, train=False, workers=WORKERS)
    eval_time = [0.0]

    def eval_fn(m):
        t0 = time.time()
        pq = summarise(evaluate(predict(m, va_dl), gt_fn, disks)).pq.iloc[0]   # threshold 0.5, min_area 0
        eval_time[0] += time.time() - t0
        return pq

    model = build_model().to(DEVICE)
    t0 = time.time()
    hist = fit(model, make_loader(tr, BATCH, train=True, workers=WORKERS), BCEDiceLoss(), epochs, LR,
               eval_fn=eval_fn, eval_every=EVAL_EVERY, log=lambda s: print(f"[{name}] {s}"))
    train_min = (time.time() - t0 - eval_time[0]) / 60

    counts = evaluate(predict(model, va_dl), gt_fn, disks, THRESHOLDS, MIN_AREAS)
    summ = summarise(counts)
    t_best, a_best = summ.pq.idxmax()
    torch.save(model.state_dict(), CKPT_DIR / f"{name}.pt")
    hist.to_csv(RUN_DIR / f"{name}_history.csv", index=False)
    counts.to_csv(RUN_DIR / f"{name}_counts.csv", index=False)
    row = {"name": name, "label": label, "mode": mode, "fold": fold, "seed": seed, "epochs": epochs, "size": S,
           "best_epoch": int(hist.loc[hist.val_pq.idxmax(), "epoch"]), "train_min": train_min,
           "pq_default": summ.loc[(0.5, 0), "pq"], "threshold": t_best, "min_area": a_best,
           **summ.loc[(t_best, a_best), ["pq", "sq", "rq"]].to_dict()}
    pd.DataFrame([row]).to_csv(RUNS_CSV, mode="a", header=not RUNS_CSV.exists(), index=False)
    del model; torch.cuda.empty_cache()
    return row

def run_all(experiments):
    done = set(pd.read_csv(RUNS_CSV).name) if RUNS_CSV.exists() else set()
    for e in experiments:
        if e["name"] in done:
            print("skip (done):", e["name"])
        else:
            run(**e)
    runs = pd.read_csv(RUNS_CSV).drop_duplicates("name", keep="last")
    return runs[runs.name.isin([e["name"] for e in experiments])].set_index("name")

## 3. Which training labels?

Notebook 02's leave-one-out test found that, as a *label*, the union or intersection of two annotators
predicts a held-out annotator no better than a single annotator does (PQ 0.359 / 0.355 vs 0.358).
A network is different: trained on "each" it sees conflicting targets but may learn the consensus itself,
while fused labels are consistent but wider (union) or merge filaments one annotator kept apart.
Only training tells which wins on PQ. One model per label type, same seed,
same fold, `full` input. `pq_default` is at threshold 0.5 / no size filter; `pq` is after tuning both.

In [ ]:
LABEL_EXPERIMENTS = [dict(name=f"unet_r34_full_{lab}", label=lab, mode="full") for lab in ("each", "union", "vote", "soft")]
label_runs = run_all(LABEL_EXPERIMENTS)
BEST_LABEL = label_runs.pq.idxmax().split("_")[-1]
print("best label:", BEST_LABEL)
label_runs[["label", "best_epoch", "train_min", "pq_default", "threshold", "min_area", "pq", "sq", "rq"]]

## 4. Which input? And what does photometric augmentation buy?

`raw` can only use geometric augmentation (section 2), so it is compared against `full` trained with the
same geometric-only augmentation; the third model isolates the effect of the photometric augmentations.

In [ ]:
INPUT_EXPERIMENTS = [
    dict(name=f"unet_r34_raw_{BEST_LABEL}_geo", label=BEST_LABEL, mode="raw"),
    dict(name=f"unet_r34_full_{BEST_LABEL}_geo", label=BEST_LABEL, mode="full", aug=GEOMETRIC),
]
input_runs = pd.concat([run_all(INPUT_EXPERIMENTS), label_runs.loc[[f"unet_r34_full_{BEST_LABEL}"]]])
input_runs[["mode", "label", "best_epoch", "pq_default", "threshold", "min_area", "pq", "sq", "rq"]]

In [ ]:
all_runs = pd.concat([label_runs, input_runs]).pipe(lambda d: d[~d.index.duplicated()])
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for name in all_runs.index:
    h = pd.read_csv(RUN_DIR / f"{name}_history.csv")
    axes[0].plot(h.epoch, h.loss, label=name)
    axes[1].plot(h.dropna(subset=["val_pq"]).epoch, h.val_pq.dropna(), marker=".", label=name)
axes[0].set(xlabel="epoch", ylabel="train loss", title="Training loss")
axes[1].set(xlabel="epoch", ylabel="val PQ (t = 0.5, no size filter)", title="Validation PQ")
axes[1].legend(fontsize=7); plt.tight_layout(); savefig("training_curves"); plt.show()

## 5. Analysis of the best model on the validation fold

In [ ]:
BEST = all_runs.pq.idxmax()
best = all_runs.loc[BEST]
T, A = float(best.threshold), int(best.min_area)
print(f"best: {BEST}  (threshold {T}, min_area {A}, PQ {best.pq:.3f})")

def load_model(name):
    m = build_model(weights=None).to(DEVICE)
    m.load_state_dict(torch.load(CKPT_DIR / f"{name}.pt", map_location=DEVICE))
    return m.eval()

model = load_model(BEST)
_, va = datasets(best.label, best["mode"], VAL_FOLD, 0)
gt_fn, disks = scorer(va)
probs = predict(model, make_loader(va, 2 * BATCH, train=False, workers=WORKERS))
counts = pd.read_csv(RUN_DIR / f"{BEST}_counts.csv")

grid = summarise(counts).pq.unstack("min_area")
fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(grid.values, cmap="viridis", aspect="auto")
ax.set(xticks=range(grid.shape[1]), xticklabels=grid.columns, yticks=range(grid.shape[0]), yticklabels=grid.index,
       xlabel="min_area (full-res px)", ylabel="threshold", title=f"Validation PQ, {BEST}")
for (i, j), v in np.ndenumerate(grid.values):
    ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7, color="w")
plt.colorbar(im); plt.tight_layout(); savefig("postprocess_grid"); plt.show()

### Against the annotator ceiling

The annotators themselves disagree (notebook 01). Inter-annotator PQ on the validation observations
with ≥ 2 sets (every ordered pair of sets) is a realistic ceiling; the model is scored on the same
observations for a like-for-like comparison.

In [ ]:
multi = [o for o in va.obs if len(va.set_ids[o]) > 1]
c = np.zeros(4)
for o in tqdm(multi, desc="annotator pairs"):
    ms = gt_fn(o)
    for i in range(len(ms)):
        for j in range(len(ms)):
            if i != j:
                m = match_instances(ms[i], ms[j]); c += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
ceiling = pq_from_counts(int(c[0]), int(c[1]), int(c[2]), c[3])
at_best = counts[(counts.threshold == T) & (counts.min_area == A)]
model_multi = summarise(at_best[at_best.obs.isin(multi)]).iloc[0]
pd.DataFrame({"annotator vs annotator": ceiling, "model vs annotators": model_multi}).loc[["pq", "sq", "rq"]].T

In [ ]:
per_img = summarise(at_best, by=["obs"]).join(folds.set_index("obs")[["site", "degraded"]])
by_site = pd.concat({s: summarise(at_best[at_best.obs.isin(g.index)]).iloc[0]
                     for s, g in per_img.groupby("site")}, axis=1).T
by_deg = pd.concat({("degraded" if d else "clean"): summarise(at_best[at_best.obs.isin(g.index)]).iloc[0]
                    for d, g in per_img.groupby("degraded")}, axis=1).T
display(pd.concat([by_site, by_deg])[["pq", "sq", "rq", "tp", "fp", "fn"]])

### What kind of errors?

Every unmatched ground-truth filament is **missed** (no predicted pixel on it), **merged** (its prediction
also covers another filament), **split** (covered by several predicted pieces) or **boundary** (one
prediction, IoU ≤ 0.5). Unmatched predictions on no filament are **spurious**. Merged and split are what
instance separation (notebook 05) can fix.

In [ ]:
err = []
for o in tqdm(va.obs, desc="error types"):
    pred = to_instances(probs[o], T, A, disks[o])
    for gt in gt_fn(o):
        err.append({"obs": o, **error_types(gt, pred)})
err = pd.DataFrame(err)
tot = err.drop(columns="obs").sum()
gt_total = tot[["matched", "missed", "merged", "split", "boundary"]].sum()
print((100 * tot[["matched", "missed", "merged", "split", "boundary"]] / gt_total).round(1).to_string(),
      f"\nspurious predictions: {int(tot.spurious)} ({tot.spurious / len(err):.2f} per annotation set)")

fig, ax = plt.subplots(figsize=(7, 3))
(100 * tot.drop("spurious") / gt_total).plot.bar(ax=ax, color=["tab:green", "tab:gray", "tab:red", "tab:orange", "tab:blue"])
ax.set(ylabel="% of GT filaments", title="Fate of ground-truth filaments (validation)"); plt.xticks(rotation=0)
plt.tight_layout(); savefig("error_types"); plt.show()

In [ ]:
def show_case(ax, o, title):
    g = tuple(c * S / FULL for c in disks[o])
    x = load_preprocessed(paths[o], disks[o], S, "full", CACHE_DIR)
    pred = cv2.resize(to_instances(probs[o], T, A, disks[o]).astype(np.float32), (S, S), interpolation=cv2.INTER_NEAREST)
    gt = va.masks(o)[0]
    ax.imshow(x, cmap="gray", vmin=0.8, vmax=1.2)
    ax.contour((gt > 0).astype(float), levels=[0.5], colors="cyan", linewidths=0.6)
    ax.contour((pred > 0).astype(float), levels=[0.5], colors="red", linewidths=0.6)
    ax.set_title(title, fontsize=9); ax.set_axis_off()

ranked = per_img[per_img.tp + per_img.fn > 0].sort_values("pq")
cases = list(ranked.index[-3:][::-1]) + list(ranked.index[:3])
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, o in zip(axes.flat, cases):
    r = per_img.loc[o]
    show_case(ax, o, f"{o} ({SITES[r.site]}{', degraded' if r.degraded else ''})\nPQ {r.pq:.2f}  TP {r.tp:.0f}  FP {r.fp:.0f}  FN {r.fn:.0f}")
fig.suptitle("Best (top) and worst (bottom) validation images. Cyan: annotation set 1, red: prediction", y=0.99)
plt.tight_layout(); savefig("best_worst_val"); plt.show()

## 6. Size, training time and end-to-end inference time

End to end means from the JPEG on disk to the instance map: load, disk detection, preprocessing,
network, post-processing. No cache is used.

In [ ]:
fill = 0.0 if best["mode"] == "raw" else 1.0

def end_to_end(path):
    t = {}
    t0 = time.perf_counter(); img = load_image(path); t["load"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    disk = find_disk(img)
    x = cv2.resize(preprocess(img, best["mode"], disk), (S, S), interpolation=cv2.INTER_AREA)
    x = model_input(x, tuple(v * S / FULL for v in disk), NORM[best["mode"]], fill)
    t["preprocess"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        p = torch.sigmoid(model(torch.from_numpy(x)[None, None].to(DEVICE)).float())[0, 0].cpu().numpy()
    t["network"] = time.perf_counter() - t0
    t0 = time.perf_counter(); inst = to_instances(p, T, A, disk); t["postprocess"] = time.perf_counter() - t0
    return inst, t

end_to_end(paths[va.obs[0]])   # warm-up (cuDNN autotuning, lazy init)
timing = pd.DataFrame([end_to_end(paths[o])[1] for o in va.obs[:30]])
timing["total"] = timing.sum(axis=1)
size = model_size(model)
summary = {
    "model": BEST, "params (M)": round(size["params_M"], 2), "weights (MB, fp32)": round(size["size_MB"], 1),
    "training time (min, excl. validation)": round(float(best.train_min), 1),
    "inference ms / image (end to end)": round(1000 * timing.total.mean(), 1),
    "inference for 180 test images (s)": round(180 * timing.total.mean(), 1),
    "device": torch.cuda.get_device_name() if DEVICE.type == "cuda" else "cpu",
}
display(pd.Series(summary).to_frame("value"))
(1000 * timing.mean()).round(1).to_frame("ms / image")

## 7. Cross-validation of the best configuration

The best configuration retrained on folds 1–4 (fold 0 is the run above), with the threshold and
`min_area` **fixed** to the values tuned on fold 0, so folds 1–4 are honest held-out estimates. Fold 0's
score is optimistic: the epoch, threshold and `min_area` were all selected on it.

In [ ]:
cfg = dict(label=best.label, mode=best["mode"])
if BEST.endswith("_geo") and best["mode"] == "full":
    cfg["aug"] = GEOMETRIC
CV_EXPERIMENTS = [dict(name=f"{BEST}_fold{f}", fold=f, **cfg) for f in sorted(folds.fold.unique()) if f != VAL_FOLD]
cv_runs = run_all(CV_EXPERIMENTS)

cv = []
for name, f in [(BEST, VAL_FOLD)] + [(e["name"], e["fold"]) for e in CV_EXPERIMENTS]:
    c = pd.read_csv(RUN_DIR / f"{name}_counts.csv")
    c = c[(c.threshold == T) & (c.min_area == A)]
    cv.append({"fold": f, **summarise(c).iloc[0][["pq", "sq", "rq"]].to_dict()})
cv = pd.DataFrame(cv).set_index("fold")
held_out = cv.drop(index=VAL_FOLD)
print(f"PQ folds 1–4: {held_out.pq.mean():.3f} ± {held_out.pq.std():.3f}   "
      f"(SQ {held_out.sq.mean():.3f} ± {held_out.sq.std():.3f}, RQ {held_out.rq.mean():.3f} ± {held_out.rq.std():.3f})")
cv

## Summary

*Fill in after the runs.*

| Question | Result | Decision |
|---|---|---|
| Training labels (each / union / vote / soft) | | |
| Input (`full` vs `raw`), photometric augmentation | | |
| Post-processing (threshold, min_area) | | |
| PQ vs annotator ceiling | | |
| Main error types | | → notebook 05 |
| Cross-validated PQ (folds 1–4) | | |
| Size / training / inference time | | → notebook 06 |